# 03 — Local-shadow Native YOLO26n Training

Full fine-tune from the pinned COCO YOLO26n on this run's S20 export with the native Ultralytics trainer. best.pt is selected on this policy's calibration split; development stays untouched until 04.

**Inputs:** v2.yaml (this folder) training args; runs/v2-YYYY-MM-DD/data; weights/yolo26n.pt.

**Outputs:** runs/v2-YYYY-MM-DD/train/{args.yaml,results.csv,weights/,complete.json} only after ALLOW_TRAINING=True.

**Prerequisites:** Run 02 separately. No upstream notebook is executed.

**Outline**

- [1. Configuration](#section-1)
- [2. Native Training](#section-2)
- [3. Training Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

RUN_DATE=None uses the newest prepared run. Native resume is explicit.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vehicle_detection.common import require_paths
from vehicle_detection.experiment_data import load_experiment
from vehicle_detection.training import train_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
EXPERIMENT = "v2"
require_paths(
    config=ROOT / f"notebooks/experiments/{EXPERIMENT}/{EXPERIMENT}.yaml",
    shared=ROOT / "configs/data.yaml",
)
config = load_experiment(ROOT, EXPERIMENT)

RUN_DATE = None
ALLOW_TRAINING = False
RESUME = False
display(pd.Series(config["training"], name="training"))

<a id="section-2"></a>

## 2. Native Training

Every training argument comes from the version YAML; changing it after a run starts requires a new run date.

In [ ]:
try:
    history = train_experiment(
        ROOT, EXPERIMENT, RUN_DATE, allow_training=ALLOW_TRAINING, resume=RESUME
    )
except FileNotFoundError as error:
    history = pd.DataFrame()
    print(error)
if not history.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
    axes[0].plot(history.epoch, history["metrics/mAP50-95(B)"] * 100, label="calibration mAP50-95")
    axes[0].set(xlabel="Epoch", ylabel="%", title="Selection metric")
    for column in ("train/box_loss", "train/cls_loss", "val/box_loss", "val/cls_loss"):
        axes[1].plot(history.epoch, history[column], label=column)
    axes[1].set(xlabel="Epoch", title="Losses")
    axes[1].legend(fontsize=8)
    plt.show()

<a id="section-3"></a>

## 3. Training Summary

Single seed; the calibration curve is a selection signal, not the reported result.

In [ ]:
if not history.empty:
    best = history.loc[history["metrics/mAP50-95(B)"].idxmax()]
    score = best["metrics/mAP50-95(B)"]
    print(f"Epochs: {len(history)} | best calibration mAP50-95 {score:.4f} @ {int(best.epoch)}")
print("Final holdout stays sealed.")